<a href="https://colab.research.google.com/github/gulzhanmsc/IB9AU/blob/main/Gen_AI_Task_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Key insights:

Key insights:

Flow Matching learns a straight-path vector field from noise to data, simpler and faster than diffusion's iterative denoising. Timestep t is encoded via sinusoidal embeddings, giving the UNet continuous temporal context. 64×64 grayscale at 50 epochs is intentionally lightweight, resolution and channel increases hit VRAM hard on T4. Inference walks 50 ODE steps from Gaussian noise to output, fewer steps = faster but lower quality.

In [ ]:
#Import
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.utils import make_grid
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import os
import glob
from PIL import Image
from google.colab import drive



In [ ]:
# MOUNT DRIVE
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# EXTRACT ZIP
!cp "/content/drive/MyDrive/Gen AI Data/floorplans_v2-20251223T170650Z-3-001.zip" /content/
!unzip /content/floorplans_v2-20251223T170650Z-3-001.zip -d /content/floorplan_images
print("Zip extracted successfully!")

Archive:  /content/floorplans_v2-20251223T170650Z-3-001.zip
replace /content/floorplan_images/floorplans_v2/floorplan_00992.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: Zip extracted successfully!


In [ ]:
# DEVICE & HYPERPARAMETERS
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
HPARAMS = {
    "img_size": 64,
    "inference_steps": 50,
    "batch_size": 32,
    "lr": 1e-4,
    "epochs": 50,
    "channels": 1,
    "num_classes": 1
}
print("Hyperparameters:", HPARAMS)

Hyperparameters: {'img_size': 64, 'inference_steps': 50, 'batch_size': 32, 'lr': 0.0001, 'epochs': 50, 'channels': 1, 'num_classes': 1}


In [ ]:
# MODEL ARCHITECTURE
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = np.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

class Block(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, up=False):
        super().__init__()
        self.time_mlp = nn.Linear(time_emb_dim, out_ch)
        if up:
            self.conv1 = nn.Conv2d(2*in_ch, out_ch, 3, padding=1)
            self.transform = nn.ConvTranspose2d(out_ch, out_ch, 4, 2, 1)
        else:
            self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
            self.transform = nn.Conv2d(out_ch, out_ch, 4, 2, 1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.bnorm1 = nn.BatchNorm2d(out_ch)
        self.bnorm2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU()

    def forward(self, x, t):
        h = self.bnorm1(self.relu(self.conv1(x)))
        time_emb = self.relu(self.time_mlp(t))
        time_emb = time_emb[(..., ) + (None, ) * 2]
        h = h + time_emb
        h = self.bnorm2(self.relu(self.conv2(h)))
        return self.transform(h)

class ConditionalUNet(nn.Module):
    def __init__(self):
        super().__init__()
        img_channels = HPARAMS["channels"]
        down_channels = (32, 64, 128)
        up_channels = (128, 64, 32)
        out_dim = img_channels
        time_emb_dim = 32
        classes = HPARAMS["num_classes"]

        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.ReLU()
        )
        self.class_emb = nn.Embedding(classes, time_emb_dim)
        self.conv0 = nn.Conv2d(img_channels, down_channels[0], 3, padding=1)
        self.downs = nn.ModuleList([Block(down_channels[i], down_channels[i+1], time_emb_dim) for i in range(len(down_channels)-1)])
        self.ups = nn.ModuleList([Block(up_channels[i], up_channels[i+1], time_emb_dim, up=True) for i in range(len(up_channels)-1)])
        self.output = nn.Conv2d(up_channels[-1], out_dim, 1)

    def forward(self, x, t_float, class_label):
        t = self.time_mlp(t_float)
        c = self.class_emb(class_label)
        t = t + c
        x = self.conv0(x)
        residuals = []
        for down in self.downs:
            x = down(x, t)
            residuals.append(x)
        for up in self.ups:
            residual = residuals.pop()
            x = torch.cat((x, residual), dim=1)
            x = up(x, t)
        return self.output(x)

In [ ]:
# FLOW MATCHING LOGIC
class FlowMatching:
    def __init__(self):
        pass

    def compute_loss(self, model, x_1, labels):
        b = x_1.shape[0]
        x_0 = torch.randn_like(x_1)
        t = torch.rand(b, device=x_1.device)
        t_view = t.view(b, 1, 1, 1)
        x_t = (1 - t_view) * x_0 + t_view * x_1
        v_target = x_1 - x_0
        v_pred = model(x_t, t, labels)
        return F.mse_loss(v_pred, v_target)

    @torch.no_grad()
    def sample(self, model, n_samples, class_label_idx, size, steps=50):
        model.eval()
        x = torch.randn((n_samples, 1, size, size)).to(device)
        labels = torch.full((n_samples,), class_label_idx, dtype=torch.long).to(device)
        dt = 1.0 / steps
        for i in range(steps):
            t_curr = torch.ones(n_samples).to(device) * (i / steps)
            v_pred = model(x, t_curr, labels)
            x = x + v_pred * dt
        model.train()
        return x

In [ ]:
# FLOORPLAN DATASET
class FloorplanDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, size=(64, 64), transform=None):
        self.size = size
        self.transform = transform
        self.image_paths = []

        for ext in ['jpg', 'jpeg', 'png']:
            self.image_paths.extend(glob.glob(os.path.join(root_dir, f'**/*.{ext}'), recursive=True))
            self.image_paths.extend(glob.glob(os.path.join(root_dir, f'*.{ext}')))

        self.image_paths = list(set(self.image_paths))

        if not self.image_paths:
            raise RuntimeError(f"No images found in {root_dir}. Check the path!")

        print(f"Found {len(self.image_paths)} floorplan images.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('L')
        label = 0
        if self.transform:
            img = self.transform(img)
        return img, label

In [ ]:
# SAVE & LOAD
def save_model(model, filename="floorplan_flow_model.pth"):
    torch.save(model.state_dict(), filename)
    print(f"Model saved to {filename}")

def load_model(filename="floorplan_flow_model.pth"):
    if not os.path.exists(filename):
        print(f"Error: {filename} not found.")
        return None
    model = ConditionalUNet().to(device)
    model.load_state_dict(torch.load(filename, map_location=device))
    model.eval()
    print(f"Model loaded from {filename}")
    return model

In [ ]:
# TRAINING
def train_floorplan_model():
    dataset = FloorplanDataset(
        root_dir='/content/floorplan_images',
        size=(HPARAMS["img_size"], HPARAMS["img_size"]),
        transform=transforms.Compose([
            transforms.Resize((HPARAMS["img_size"], HPARAMS["img_size"])),
            transforms.ToTensor()
        ])
    )
    dataloader = DataLoader(dataset, batch_size=HPARAMS["batch_size"], shuffle=True)

    model = ConditionalUNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=HPARAMS["lr"])
    flow = FlowMatching()

    print(f"Starting training for {HPARAMS['epochs']} epochs...")
    for epoch in range(HPARAMS['epochs']):
        pbar = tqdm(dataloader)
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)
            loss = flow.compute_loss(model, images, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            pbar.set_description(f"Epoch {epoch+1} | Loss: {loss.item():.4f}")

    return model, flow

In [ ]:
# GENERATE FLOORPLANS
def generate_floorplans(model, flow, n_samples=8):
    print("\nGenerating synthetic floorplans...")
    generated = flow.sample(
        model,
        n_samples=n_samples,
        class_label_idx=0,
        size=HPARAMS["img_size"],
        steps=HPARAMS["inference_steps"]
    )

    grid = make_grid(generated, nrow=4, padding=2, normalize=True)
    plt.figure(figsize=(12, 6))
    plt.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
    plt.title(f"Generated Synthetic Floorplans ({n_samples} samples)")
    plt.axis('off')
    plt.show()

def generate_single_floorplan(model, flow):
    print("\nGenerating single floorplan...")
    sample_tensor = flow.sample(
        model,
        n_samples=1,
        class_label_idx=0,
        size=HPARAMS["img_size"],
        steps=HPARAMS["inference_steps"]
    )
    img = sample_tensor[0].cpu().permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min())
    plt.figure(figsize=(4, 4))
    plt.imshow(img, cmap='gray')
    plt.title("Generated Floorplan")
    plt.axis('off')
    plt.show()

In [ ]:
# MAIN
trained_model, flow_manager = train_floorplan_model()
save_model(trained_model, "floorplan_flow_model.pth")
generate_floorplans(trained_model, flow_manager, n_samples=8)
generate_single_floorplan(trained_model, flow_manager)

RuntimeError: No images found in /content/floorplan_images. Check the path!